# Phase 5-B3-B｜Controlled Real Research Execution Engineering

**Notebook 版本：`PHASE5B3B-EXECUTION-v1`**　｜　Code pin `d9afac9`　｜　分支 `feature/phase5b3b-controlled-real-research`

**只做工程驗收：不消耗正式研究額度、不執行新的市場回測、不讀 Holdout、不建立或核准授權；正式 Runner 維持 DISABLED。**
- 驗收 REAL_GATED 受控執行路徑（啟用 gate A01–A10），全部在隔離帳本上：mock runner，以及既有 ResearchOnlyRunner＋Core `b16179a` 在 Core 產生的 **synthetic** snapshot 上（含 Core receipt renewal）。
- 即時檢查 repo 的正式帳本：沒有任何授權指名 executor → 真實路徑無法建立（fail closed）。
- 四層分開報告：Engineering Readiness／Research Authorization／Real Execution／Research Result Acceptance。
- 輸出：`MyDrive/Quant_Research/strategy-research-agent/phase5b3b/<YYYYMMDD_HHMMSS>_<pin7>/`，不可覆寫，含 JUnit 與 `ARCHIVE_INDEX.json`，並比對下載的 ZIP。

**請在全新的 Colab runtime 開啟後 Run all。** Google Drive 授權在**第 1 格**；看到 `DRIVE: MOUNTED` 後就可以離開。

In [ ]:
# --- 1. 設定 + 人工核可（全部集中在這一格；任何一項失敗立即 STOP）---
NOTEBOOK_VERSION = "PHASE5B3B-EXECUTION-v1"
NOTEBOOK_PATH    = "notebooks/Phase5B3B_Controlled_Execution_Acceptance.ipynb"
AGENT_REPO = "https://github.com/hh4832/strategy-research-agent.git"
BRANCH     = "feature/phase5b3b-controlled-real-research"
PIN_SHA    = "d9afac9d079d8cf5c3f3f2e1a3425bb1e3ff4e90"
CORE_SHA   = "b16179a7cc077c36046574769a32c367fa476cae"
DRIVE_BASE = "/content/drive/MyDrive/Quant_Research/strategy-research-agent"
AGENT_ROOT = "/content/strategy-research-agent-phase5b3b"

import json, os, re, subprocess, sys
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo
assert sys.version_info >= (3, 11), sys.version
CLEAN_RUNTIME = not any(m in sys.modules for m in ("strategy_agent", "market_research_core"))
AGENT_ROOT = Path(AGENT_ROOT)
def git(*a, cwd=None):
    r = subprocess.run(["git", *a], cwd=cwd, text=True, capture_output=True)
    if r.returncode: raise RuntimeError("git " + a[0] + " failed: " + r.stderr[-400:])
    return r.stdout.strip()
ENV = {k: v for k, v in os.environ.items() if k not in ("GH_TOKEN", "GITHUB_TOKEN")}
def py(args, **kw):
    return subprocess.run([sys.executable, *args], cwd=AGENT_ROOT, text=True, capture_output=True, env=ENV, **kw)
import importlib.util
CORE_PREINSTALLED = importlib.util.find_spec("market_research_core") is not None
print(NOTEBOOK_VERSION, "| pin", PIN_SHA[:7], "| python", sys.version.split()[0], "| clean runtime:", CLEAN_RUNTIME)
assert not CORE_PREINSTALLED, ("STOP: 此 runtime 已安裝過 market_research_core（不是全新 runtime）；"
                               "請 執行階段 → 中斷連線並刪除執行階段，再 Run all")

from google.colab import drive
drive.mount("/content/drive")
CLOUD = Path(DRIVE_BASE).is_dir()
for p in ["/content/drive/MyDrive", "/content/drive/MyDrive/Quant_Research", DRIVE_BASE]:
    print(Path(p).is_dir(), p)
assert CLOUD, ("STOP: Drive 未掛載或找不到 " + DRIVE_BASE +
               "（授權視窗是否完成？登入的是放 Quant_Research 的帳號嗎？）")
print("DRIVE: MOUNTED →", DRIVE_BASE)

In [ ]:
# --- 2. Version Sync ---
assert CLEAN_RUNTIME, "STOP: 此 runtime 已載入 strategy_agent / market_research_core；請 Runtime → Restart session 後 Run all"
if not AGENT_ROOT.exists():
    git("clone", "--quiet", AGENT_REPO, str(AGENT_ROOT)); git("checkout", "--quiet", "--detach", PIN_SHA, cwd=AGENT_ROOT)
remote = git("remote", "get-url", "origin", cwd=AGENT_ROOT)
assert remote.rstrip("/").removesuffix(".git") == AGENT_REPO.removesuffix(".git"), "STOP: wrong remote " + remote
head = git("rev-parse", "HEAD", cwd=AGENT_ROOT)
status = git("status", "--porcelain", "--untracked-files=all", cwd=AGENT_ROOT)
git("fetch", "--quiet", "origin", BRANCH, cwd=AGENT_ROOT)
remote_head = git("rev-parse", "FETCH_HEAD", cwd=AGENT_ROOT)
on_branch = subprocess.run(["git", "merge-base", "--is-ancestor", PIN_SHA, remote_head], cwd=AGENT_ROOT).returncode == 0
nb = subprocess.run(["git", "show", f"{remote_head}:{NOTEBOOK_PATH}"], cwd=AGENT_ROOT, text=True, capture_output=True)
gh_version = (re.search(r'NOTEBOOK_VERSION = \\"([^\\"]+)\\"', nb.stdout) or [None, None])[1] if nb.returncode == 0 else None
after_pin = git("diff", "--name-only", PIN_SHA, remote_head, cwd=AGENT_ROOT).splitlines() if remote_head != PIN_SHA else []
SYNC = {"notebook_version": NOTEBOOK_VERSION, "notebook_version_on_github_head": gh_version, "github_remote_head": remote_head,
        "checkout_head": head, "pinned_sha": PIN_SHA, "pin_on_branch": on_branch, "working_tree_clean": status == "",
        "files_changed_after_pin": after_pin}
print(json.dumps(SYNC, indent=2))
assert head == PIN_SHA and status == "" and on_branch, "STOP: checkout does not match the pin"
assert gh_version in (None, NOTEBOOK_VERSION), f"STOP: 這是舊 notebook；GitHub 上是 {gh_version}"
code_after_pin = [p for p in after_pin if not p.startswith(("docs/", "notebooks/"))]
assert not code_after_pin, f"STOP: pin 之後 GitHub 上有程式／治理變更 {code_after_pin}；需要新的 pin"
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(AGENT_ROOT) + "[test,research]"], check=True)
print("VERSION SYNC: PASS")

In [ ]:
# --- 3. 完整回歸（尚未安裝 Core）：任何非零結束碼都 STOP ---
t = py(["-m", "pytest", "-q", "-p", "no:cacheprovider"], timeout=3600)
REG = {"returncode": t.returncode, "summary": t.stdout.strip().splitlines()[-1] if t.stdout.strip() else "", "core": "absent"}
print(t.stdout[-1200:]); print(REG)
assert t.returncode == 0, "STOP: regression failed"

In [ ]:
# --- 4. 安裝 Core b16179a（唯讀）並跑 5-B3-B / 5-B2 / 5-B1 suites（junit；B3B-01..09 的測試證據）---
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                f"market-research-core @ git+https://github.com/hh4832/market-research-core@{CORE_SHA}"], check=True)
JUNIT = Path("/content/b3b_junit.xml")
import glob
suites = sorted(glob.glob(str(AGENT_ROOT / "tests/test_phase5b3b_*.py")) + glob.glob(str(AGENT_ROOT / "tests/test_phase5b2_*.py"))
                + glob.glob(str(AGENT_ROOT / "tests/test_phase5b1*.py")))
b = py(["-m", "pytest", "-q", "-p", "no:cacheprovider", *suites, "--junitxml", str(JUNIT)], timeout=3600)
B2 = {"returncode": b.returncode, "summary": b.stdout.strip().splitlines()[-1] if b.stdout.strip() else "", "core": CORE_SHA}
print(b.stdout[-1500:]); print(B2)
assert b.returncode == 0, "STOP: governance / controlled-execution suites failed"
assert git("status", "--porcelain", "--untracked-files=all", cwd=AGENT_ROOT) == "", "STOP: tests modified the checkout"

In [ ]:
# --- 5. B3B-01..B3B-10 acceptance（全新 subprocess；Drive 已在第 1 格授權）---
assert CLOUD and Path(DRIVE_BASE).is_dir(), "STOP: Drive 已不在（第 1 格授權後被中斷？）請重跑"
OUT = Path(DRIVE_BASE) / "phase5b3b" / f"{datetime.now(ZoneInfo('Asia/Taipei')):%Y%m%d_%H%M%S}_{PIN_SHA[:7]}"
assert not OUT.exists(), "STOP: output exists (never overwritten)"
deps = py(["-m", "pip", "freeze"]).stdout.splitlines()
pins = [d for d in deps if re.match(r"(market-research-core|strategy-research-agent|pandas|pyarrow|numpy|jsonschema|pytest)\b", d, re.I)]
cj = Path("/content/b3b_context.json"); cj.write_text(json.dumps({"notebook": NOTEBOOK_VERSION, "version_sync": SYNC,
    "clean_colab_runtime": CLEAN_RUNTIME, "cloud": CLOUD, "python": sys.version.split()[0], "regression": REG,
    "suites": B2, "dependency_pins": pins,
    "executor": "this Colab runtime (engineering acceptance only; no real execution)"}))
rc = 0 if (REG["returncode"] == 0 and B2["returncode"] == 0) else 1
cmd = ["-m", "strategy_agent.execution.real_acceptance", "--out-dir", str(OUT), "--junit", str(JUNIT), "--tests-rc", str(rc),
       "--context-json", str(cj), "--cloud"]
r = py(cmd, timeout=1800)
print(r.stdout[-5000:]); print(r.stderr[-2000:])
assert r.returncode == 0, "STOP: acceptance crashed"
RES = json.loads(r.stdout[r.stdout.index("{"):])
print("VERDICT:", RES["verdict"]["verdict"], "| gates:", RES["gates"]); print("LAYERS:", RES["layers"])
assert git("status", "--porcelain", "--untracked-files=all", cwd=AGENT_ROOT) == "", "STOP: acceptance modified the checkout"

In [ ]:
# --- 6. ZIP + 封存一致性驗證（ARCHIVE_INDEX 涵蓋全部檔案；之後不再寫入）---
import zipfile
zp = Path("/content") / (OUT.name + "_phase5b3b_execution.zip")
with zipfile.ZipFile(zp, "w") as z:
    for p in sorted(OUT.iterdir()):
        if p.is_file(): z.write(p, p.name)
v = py(["-m", "strategy_agent.execution.real_acceptance", "--verify-archive", str(OUT), "--zip", str(zp)])
print(v.stdout); assert v.returncode == 0, "STOP: archive / ZIP verification failed"
print((OUT / "run_info.txt").read_text())
try:
    from google.colab import files as F; F.download(str(zp))
except Exception as e:                       # noqa: BLE001
    print("download skipped:", e, "| zip:", zp)